# Aufbereitung Bevölkerungsdaten

Quelle: BFS, *Ständige und nichtständige Wohnbevölkerung nach Jahr, Kanton, Bevölkerungstyp, Staatsangehörigkeit, Geburtsstaat und Geschlecht* (px-x-0103010000_499), 2010 bis 2025.

**Warum nicht `pyaxis` wie bei den Kriminalitätsdaten?** Die Datei enthält 109 Mio. Zellen (jede Staatsangehörigkeit × jeder Geburtsstaat). `pyaxis` baut daraus ein riesiges DataFrame und braucht über 6 GB RAM. Darum lesen wir die Zahlen direkt mit numpy ein (ca. 1.2 GB, 30 Sekunden) und behalten nur, was wir brauchen:

- Geburtsstaat: nur Total
- Staatsangehörigkeit: Total, Schweiz und daraus berechnet Ausland
- alle Jahre, Kantone, Bevölkerungstypen und Geschlechter

Gespeichert wird als **Parquet** (kleiner und schneller als CSV).

In [ ]:
import re
import numpy as np
import pandas as pd

## 1. PX-Datei einlesen

In [ ]:
text = open("Bevolkerungsdaten.px", encoding="utf-8-sig").read()
kopf, daten = text.split("DATA=", 1)
del text

# Dimensionen (Reihenfolge der Daten) und ihre Ausprägungen aus dem Kopf lesen
liste = lambda s: re.findall(r'"([^"]*)"', s)
dims = liste(re.search(r"STUB=(.*?);", kopf, re.S).group(1)) + liste(re.search(r"HEADING=(.*?);", kopf, re.S).group(1))
werte = {d: liste(re.search(r'VALUES\("%s"\)=(.*?);' % re.escape(d), kopf, re.S).group(1)) for d in dims}
{d: len(v) for d, v in werte.items()}

In [ ]:
# Zahlen direkt als numpy-Array einlesen (fehlende Werte wie "..." werden zu NaN)
daten = re.sub(r'"[^"]*"', " nan ", daten).rstrip().rstrip(";")
arr = np.fromstring(daten, sep=" ")
del daten

arr = arr.reshape([len(werte[d]) for d in dims])
print(arr.shape, "fehlende Werte:", np.isnan(arr).sum())

## 2. Relevante Teile auswählen

In [ ]:
i_st = [werte["Staatsangehörigkeit"].index(x) for x in ["Staatsangehörigkeit - Total", "Schweiz"]]
i_gb = werte["Geburtsstaat"].index("Geburtsstaat - Total")
teil = arr[:, :, :, i_st, i_gb, :]  # Form: Jahr, Kanton, Bevölkerungstyp, Staatsangehörigkeit, Geschlecht
del arr

index = pd.MultiIndex.from_product(
    [werte["Jahr"], werte["Kanton"], werte["Bevölkerungstyp"], ["Total", "Schweiz"], werte["Geschlecht"]],
    names=["Jahr", "Kanton", "Bevölkerungstyp", "Staatsangehörigkeit", "Geschlecht"],
)
bev = pd.Series(teil.ravel(), index=index, name="DATA").unstack("Staatsangehörigkeit")
bev["Ausland"] = bev["Total"] - bev["Schweiz"]
bev = bev.stack().rename("DATA").reset_index()

bev["Jahr"] = bev["Jahr"].astype(int)
for c in ["Kanton", "Bevölkerungstyp", "Staatsangehörigkeit", "Geschlecht"]:
    bev[c] = bev[c].astype("category")
print(bev.shape)
bev.head(10)

## 3. Plausibilitätscheck

In [ ]:
# Ständige Wohnbevölkerung Neuchâtel (beide Geschlechter)
bev[(bev["Kanton"] == "Neuchâtel") & (bev["Bevölkerungstyp"] == "Ständige Wohnbevölkerung")
    & (bev["Geschlecht"] == "Geschlecht - Total")].pivot_table(index="Jahr", columns="Staatsangehörigkeit", values="DATA", observed=True)

In [ ]:
# Summe der Kantone muss dem Total Schweiz entsprechen
s = bev[bev["Geschlecht"] == "Geschlecht - Total"].pivot_table(index=["Jahr", "Bevölkerungstyp", "Staatsangehörigkeit"], columns="Kanton", values="DATA", observed=True)
print("Grösste Abweichung Kantone vs. Schweiz:", (s.drop(columns="Schweiz").sum(axis=1) - s["Schweiz"]).abs().max())

## 4. Als Parquet speichern

In [ ]:
bev.to_parquet("bevoelkerung.parquet", index=False)
# Später einlesen mit: bev = pd.read_parquet("bevoelkerung.parquet")

## 5. Kriminalitätsdaten von CSV in Parquet umwandeln

`beschuldigte.csv` ist 4.6 GB gross. Die Textspalten werden beim Einlesen direkt als Kategorie gespeichert, das spart viel Arbeitsspeicher. Danach reicht in den anderen Notebooks `pd.read_parquet("beschuldigte.parquet")`.

Diese Zelle muss nur **einmal** laufen.

In [ ]:
import os
spalten = ["Straftat", "Kanton", "Aufenthaltsgruppe", "Geschlecht", "Altersklasse"]
df_b = pd.read_csv("beschuldigte.csv", dtype={c: "category" for c in spalten})
df_b.to_parquet("beschuldigte.parquet", index=False)

print(df_b.shape)
print(f"CSV:     {os.path.getsize('beschuldigte.csv') / 1e9:.2f} GB")
print(f"Parquet: {os.path.getsize('beschuldigte.parquet') / 1e9:.2f} GB")